# How to manually create a simulation
In this notebook we are going to deeply understand sat_com_topology library. In this notebook we will create a simulation without configuration. We will try to connect the ISS (International SPace Station) to a ground station.
       
User --- ISS --- Ground Station



In [27]:
from datetime import datetime

from sat_com_adapter.adapters import NetworkXAdapter, CesiumAdapter
from sat_com_builder.configuration_manager import EmptyConfigurationManager
from sat_com_model.models import create_satellite, create_ground_station, create_user_terminal
from sat_com_trajectopy.orbital_models import PyOrbitalModel

## Generate an empty simulation

In [28]:
empty_configuration_manager = EmptyConfigurationManager()

simulation_manager = empty_configuration_manager.load_simulation()

## Configure a simulation date

In [29]:
date_string = "15/03/2024 21:10:16"
date_format = "%d/%m/%Y %H:%M:%S"

date_simulation = datetime.strptime(date_string, date_format)

## Configure CESIUM TOKEN

In [30]:
CESIUM_TOKEN = "<YOUR_CESIUM_TOKEN>"

## Create the satellite (ISS)
### Define the TLE

In [31]:

tle = {
    "satellite_name": "ISS (ZARYA)",      
    "line1": "1 25544U 98067A   14273.50403866  .00012237  00000-0  21631-3 0  1790",
    'line2:': "2 25544  51.6467 297.5710 0002045 126.1182  27.2142 15.50748592907666",
}

### Create the satellite

In [32]:
satellite = create_satellite(tle["satellite_name"], 0)

### Set the Movement Model

In [33]:
orbital_model = PyOrbitalModel(tle, date_simulation)
satellite.set_movement_model(orbital_model)

### Add the Satellite to the simulation

In [34]:
simulation_manager.add_satellite(satellite)

## Configure Ground Station

In [35]:
ground_station = create_ground_station(1)
ground_station.set_position(
    longitude= 2.349014,
    latitude= 48.864716,
    altitude= 0,
)
simulation_manager.add_ground_station(ground_station)

## Configure User Terminal

In [36]:
user_terminal = create_user_terminal(0, "wanna_connect")
user_terminal.set_position(
    longitude= 1.444000,
    latitude= 43.604500,
    altitude= 0,
)
simulation_manager.add_user_terminals(user_terminal)

## Create links

In [37]:
simulation_manager.create_ground_station_link_connection(satellite, ground_station)
simulation_manager.create_user_terminal_link_connection(satellite, user_terminal)

## Export simulation

In [38]:
networkx_adapter = NetworkXAdapter(simulation_manager, "results")
networkx_adapter.create_full_networkx_graph(export_object_position=True)
networkx_adapter.adapt()

In [39]:
cesium_adapter = CesiumAdapter(simulation_manager, "results")
cesium_adapter.build_renderer_simulation_with_links(CESIUM_TOKEN)
cesium_adapter.adapt()